# Module 2 — Topic 3: Cleaning Data
## Live Demo Notebook

**Scenario:** You've loaded `orders.csv` and your `info()` check revealed several problems. Now it's time to fix them — systematically, in the right order, and with full validation at the end.

**Problems we know exist (from Topic 2):**
1. `phone` — 8 missing values
2. `total_amount` — 6 missing values
3. `unit_price` — stored as a string with `₦` prefix (should be float)
4. `state` — inconsistent capitalisation (`Lagos`, `lagos`, `LAGOS`)
5. Duplicate rows — 4 exact duplicates at the bottom of the file

**Cleaning order:** Inspect → Missing → Duplicates → Types → Strings → Validate

---

## Part 1 — Load and Inspect

In [ ]:
import pandas as pd
import numpy as np

# Load raw data
df = pd.read_csv("../../../data/orders.csv", encoding="utf-8-sig")

print(f"Raw dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

In [ ]:
# Full structural inspection
df.info()

In [ ]:
# Missing value counts and percentages
missing_counts = df.isnull().sum()
missing_pct    = (missing_counts / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    "missing_count":   missing_counts,
    "missing_percent": missing_pct
})

print("Columns with missing values:")
print(missing_report[missing_report["missing_count"] > 0])

In [ ]:
# Duplicate check
print(f"Duplicate rows: {df.duplicated().sum()}")
print()
print("The duplicate rows:")
df[df.duplicated(keep=False)].sort_values("order_id").head(10)

In [ ]:
# Type issues — unit_price should be numeric
print("unit_price dtype:", df["unit_price"].dtype)
print("unit_price samples:", df["unit_price"].head(8).tolist())
print()

# Casing issue — state column
print("Unique state values:")
print(sorted(df["state"].unique()))

---
## Part 2 — Make a Copy (Never Modify the Original)

In [ ]:
# Always work on a copy — never modify the original DataFrame
df_clean = df.copy()

print("Working copy created:", df_clean.shape)
print("Original preserved:  ", df.shape)

---
## Part 3 — Fix Missing Values

In [ ]:
# phone: can't recover a missing phone number
# Fill with a placeholder — keeps the row, flags the gap
df_clean["phone"] = df_clean["phone"].fillna("Not provided")

print("Missing phone values after fill:", df_clean["phone"].isnull().sum())
print("Sample phone values:")
print(df_clean["phone"].value_counts().head())

In [ ]:
# total_amount: this is our key revenue metric
# We cannot analyse revenue with missing amounts — drop those rows

before_shape = df_clean.shape
df_clean = df_clean.dropna(subset=["total_amount"])
after_shape  = df_clean.shape

print(f"Before dropna: {before_shape[0]} rows")
print(f"After dropna:  {after_shape[0]} rows")
print(f"Rows removed:  {before_shape[0] - after_shape[0]}")
print(f"Missing total_amount: {df_clean['total_amount'].isnull().sum()}")

---
## Part 4 — Remove Duplicates

In [ ]:
before_shape = df_clean.shape
df_clean = df_clean.drop_duplicates()
after_shape  = df_clean.shape

print(f"Before drop_duplicates: {before_shape[0]} rows")
print(f"After drop_duplicates:  {after_shape[0]} rows")
print(f"Duplicates removed:     {before_shape[0] - after_shape[0]}")

In [ ]:
# Confirm zero duplicates remain
print("Remaining duplicates:", df_clean.duplicated().sum())

# Also check for duplicate order_ids (same ID, slightly different rows)
print("Duplicate order_ids: ", df_clean.duplicated(subset=["order_id"]).sum())

---
## Part 5 — Fix Data Types

In [ ]:
# unit_price: strip the ₦ symbol, then convert to float
print("Before — dtype:", df_clean["unit_price"].dtype)
print("Before — sample:", df_clean["unit_price"].head(5).tolist())

# Step 1: remove the ₦ symbol
df_clean["unit_price"] = df_clean["unit_price"].str.replace("₦", "", regex=False)

# Step 2: convert to numeric (errors='coerce' turns any remaining bad values to NaN)
df_clean["unit_price"] = pd.to_numeric(df_clean["unit_price"], errors="coerce")

print()
print("After — dtype:", df_clean["unit_price"].dtype)
print("After — sample:", df_clean["unit_price"].head(5).tolist())
print("Any new missing values?", df_clean["unit_price"].isnull().sum())

In [ ]:
# order_date: convert string to datetime
print("Before — dtype:", df_clean["order_date"].dtype)
print("Before — sample:", df_clean["order_date"].head(3).tolist())

df_clean["order_date"] = pd.to_datetime(df_clean["order_date"])

print()
print("After — dtype:", df_clean["order_date"].dtype)
print("After — sample:", df_clean["order_date"].head(3).tolist())

In [ ]:
# Now we can extract date components — unlocked by the datetime conversion
df_clean["order_month"] = df_clean["order_date"].dt.month
df_clean["order_year"]  = df_clean["order_date"].dt.year

print("Orders per month:")
print(df_clean["order_month"].value_counts().sort_index())

---
## Part 6 — Clean String Columns

In [ ]:
# state: standardise to Title Case
print("Before — unique state values:")
print(sorted(df_clean["state"].unique()))
print(f"Total unique values: {df_clean['state'].nunique()}")

In [ ]:
df_clean["state"] = df_clean["state"].str.strip().str.title()

print("After — unique state values:")
print(sorted(df_clean["state"].unique()))
print(f"Total unique values: {df_clean['state'].nunique()}")

In [ ]:
# Show the impact on groupby — this is WHY casing matters
print("=== BEFORE cleaning (on raw df) ===")
print(df.groupby("state")["total_amount"].sum().sort_values(ascending=False).head(10))

print()
print("=== AFTER cleaning (on df_clean) ===")
print(df_clean.groupby("state")["total_amount"].sum().sort_values(ascending=False).head(10))

---
## Part 7 — Add Useful Columns, Drop Unnecessary Ones

In [ ]:
# Add a revenue tier label — useful for segmentation
def assign_tier(amount):
    if amount > 30000:
        return "Premium"
    elif amount > 10000:
        return "Mid-range"
    else:
        return "Standard"

df_clean["order_tier"] = df_clean["total_amount"].apply(assign_tier)

print("Order tier distribution:")
print(df_clean["order_tier"].value_counts())

In [ ]:
# Add VAT-inclusive amount (7.5% Nigerian VAT rate)
df_clean["amount_with_vat"] = (df_clean["total_amount"] * 1.075).round(2)

print("Sample — total_amount vs amount_with_vat:")
df_clean[["order_id", "total_amount", "amount_with_vat"]].head(6)

---
## Part 8 — Full Validation

In [ ]:
print("===== CLEANING VALIDATION REPORT =====")
print()
print(f"Original shape: {df.shape}")
print(f"Clean shape:    {df_clean.shape}")
print(f"Rows removed:   {df.shape[0] - df_clean.shape[0]} (6 missing + 4 duplicates)")

In [ ]:
remaining_missing = df_clean.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if len(remaining_missing) == 0:
    print("Missing values: NONE ✓")
else:
    print("Missing values still present:")
    print(remaining_missing)

In [ ]:
dupe_count = df_clean.duplicated().sum()
print(f"Duplicate rows: {dupe_count} {'✓' if dupe_count == 0 else '✗ — still has duplicates!'}")

In [ ]:
print("Data types:")
print(df_clean.dtypes)
print()
print("unit_price is numeric:", pd.api.types.is_numeric_dtype(df_clean["unit_price"]))
print("order_date is datetime:", pd.api.types.is_datetime64_any_dtype(df_clean["order_date"]))

In [ ]:
print("Unique state values after cleaning:")
print(sorted(df_clean["state"].unique()))
print(f"Count: {df_clean['state'].nunique()} (expected 10)")

In [ ]:
print("Value ranges:")
print(f"  total_amount — min: NGN {df_clean['total_amount'].min():,.0f}, "
      f"max: NGN {df_clean['total_amount'].max():,.0f}")
print(f"  unit_price   — min: NGN {df_clean['unit_price'].min():,.0f}, "
      f"max: NGN {df_clean['unit_price'].max():,.0f}")
print(f"  quantity     — min: {df_clean['quantity'].min()}, "
      f"max: {df_clean['quantity'].max()}")
print(f"  order_date   — from: {df_clean['order_date'].min().date()}, "
      f"to: {df_clean['order_date'].max().date()}")

---
## Part 9 — Save the Clean Dataset

In [ ]:
df_clean.to_csv("../../../data/orders_clean.csv", index=False, encoding="utf-8-sig")

print(f"orders_clean.csv saved: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")

# Quick reload to verify
verify = pd.read_csv("../../../data/orders_clean.csv", encoding="utf-8-sig")
print(f"Reload verified:        {verify.shape[0]} rows x {verify.shape[1]} columns")
print("Columns:", verify.columns.tolist())

---
## Summary

**What we fixed in `orders.csv`:**

| Problem | Fix | Method |
|---|---|---|
| 8 missing `phone` values | Filled with 'Not provided' | `fillna()` |
| 6 missing `total_amount` values | Dropped those rows | `dropna(subset=)` |
| 4 duplicate rows | Removed | `drop_duplicates()` |
| `unit_price` stored as string | Stripped ₦, converted to float | `.str.replace()` + `pd.to_numeric()` |
| `order_date` stored as string | Converted to datetime | `pd.to_datetime()` |
| `state` inconsistent casing | Standardised to Title Case | `.str.title()` |
| Added `order_month`, `order_year` | Extracted from datetime | `.dt.month`, `.dt.year` |
| Added `order_tier` | Revenue segmentation | `.apply()` |
| Added `amount_with_vat` | 7.5% VAT calculation | Element-wise multiplication |

**Result:** 500 rows → 490 clean rows. Every column has the correct dtype. No missing values. No duplicates. `orders_clean.csv` is ready for analysis in Topics 4–6.

**Next — Topic 4:** Transforming Data — filtering, sorting, groupby, merging.